In [0]:
# 1. We are calling the table you just uploaded
# Replace 'main.default.ibm_raw_data' with whatever you named it in the UI
raw_data = spark.table("workspace.default.ibm_raw_data")

# Let's see if the data loaded correctly
display(raw_data.limit(5))

Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [0]:
from pyspark.sql.functions import col, when, lit

# We are creating a 'Silver' version of the data with new Banking Metrics
silver_df = raw_data.select(
    "EmployeeNumber", 
    "Age", 
    "Department", 
    "JobRole", 
    "MonthlyIncome", 
    "Attrition",
    "TotalWorkingYears"
).withColumn(
    # In banking, replacing an employee costs roughly 30% of their annual salary
    "attrition_cost", 
    when(col("Attrition") == "Yes", col("MonthlyIncome") * 12 * 0.30)
    .otherwise(0)
)

# Display our new 'Fiscal' column
display(silver_df.select("EmployeeNumber", "Department", "Attrition", "attrition_cost").filter(col("Attrition") == "Yes"))

EmployeeNumber,Department,Attrition,attrition_cost
1,Sales,Yes,21574.8
4,Research & Development,Yes,7524.0
19,Research & Development,Yes,7300.8
27,Sales,Yes,12265.199999999999
31,Research & Development,Yes,10656.0
33,Research & Development,Yes,14108.4
42,Sales,Yes,7509.599999999999
45,Research & Development,Yes,8254.8
47,Sales,Yes,9658.8
55,Research & Development,Yes,8254.8


In [0]:
# 1. Create the Fact Table (The 'Numbers' table)
fact_attrition = silver_df.select(
    col("EmployeeNumber").alias("emp_id"),
    col("MonthlyIncome"),
    col("attrition_cost").alias("fiscal_impact_usd")
)

# 2. Create the Dimension Table (The 'Details' table)
dim_employee = silver_df.select(
    col("EmployeeNumber").alias("emp_id"),
    col("Department"),
    col("JobRole"),
    col("Age"),
    col("TotalWorkingYears")
)

# Check the count to be sure we have everything (No limit here!)
print(f"Total rows in Fact Table: {fact_attrition.count()}")

Total rows in Fact Table: 1470


In [0]:
# Save Fact table to Unity Catalog
fact_attrition.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_fact_attrition")

# Save Dimension table to Unity Catalog
dim_employee.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_dim_employee")

print("Tables saved successfully to Unity Catalog!")

Tables saved successfully to Unity Catalog!


In [0]:
# Pehle Fact Table ko display karte hain
df_fact = spark.table("workspace.default.gold_fact_attrition")
display(df_fact)

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?


emp_id,MonthlyIncome,fiscal_impact_usd
1,5993,21574.8
2,5130,0.0
4,2090,7524.0
5,2909,0.0
7,3468,0.0
8,3068,0.0
10,2670,0.0
11,2693,0.0
12,9526,0.0
13,5237,0.0


In [0]:
# Dimension Table ko display karke download karne ke liye
df_dim = spark.table("workspace.default.gold_dim_employee")
display(df_dim)

emp_id,Department,JobRole,Age,TotalWorkingYears
1,Sales,Sales Executive,41,8
2,Research & Development,Research Scientist,49,10
4,Research & Development,Laboratory Technician,37,7
5,Research & Development,Research Scientist,33,8
7,Research & Development,Laboratory Technician,27,6
8,Research & Development,Laboratory Technician,32,8
10,Research & Development,Laboratory Technician,59,12
11,Research & Development,Laboratory Technician,30,1
12,Research & Development,Manufacturing Director,38,10
13,Research & Development,Healthcare Representative,36,17


In [0]:
display(spark.catalog.listTables())

name,catalog,namespace,description,tableType,isTemporary
gold_dim_employee,workspace,List(default),"The table contains employee demographic and job-related information. It includes data such as employee ID, department, job role, age, and total working years. Use cases include workforce analysis, departmental performance evaluations, and understanding employee tenure.",MANAGED,false
gold_fact_attrition,workspace,List(default),"The table contains data related to employee attrition, including employee identifiers and financial aspects. It can be used for analysis of employee turnover and its fiscal impact, helping to understand trends in attrition, assess the financial consequences, and make informed decisions regarding retention strategies.",MANAGED,false
ibm_raw_data,workspace,List(default),"The table contains employee-related data, including demographics, job details, and satisfaction metrics. It can be used to analyze employee attrition, assess workplace environment factors, and evaluate job satisfaction levels. Key data points include age, gender, job role, income, and various satisfaction scores, which may assist in HR analytics or workforce planning.",MANAGED,false


In [0]:
# from pyspark.sql import functions as F

df = spark.table("workspace.default.gold_fact_attrition")

quartiles = df.select(
    F.percentile_approx("MonthlyIncome", [0.25, 0.75]).alias("quartiles")
).collect()[0]["quartiles"]

q1 = quartiles[0]
q3 = quartiles[1]

print(f"Actual Q1 (25th Percentile): {q1}")
print(f"Actual Q3 (75th Percentile): {q3}")

print(f"Took round-off 4000 and Q3 8000.")

Actual Q1 (25th Percentile): 2911
Actual Q3 (75th Percentile): 8380
Took round-off 4000 and Q3 8000.
